<a href="https://colab.research.google.com/github/kamathvk1982/GAI601/blob/main/AssignmentFour-EthicsAndExplainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment 4: Ethics and Explainability**

Author(s): Vinayak Kamath

Course/Subject: GAI 601 Machine Learning

Date of Submission: May 10th, 2026

**Introduction**

In the context of machine learning and artificial intelligence (AI), ethics and explainability are increasingly becoming core considerations. As AI models are deployed in real-world applications, it is essential to ensure that they are not only accurate and efficient but also fair, transparent, and accountable. The complexity of modern machine learning models, such as deep neural networks and ensemble models, often makes them "black boxes" – decisions made by these models are not easily interpretable by humans. This lack of transparency can lead to ethical concerns, particularly in sensitive applications like healthcare, finance, and criminal justice.

Explainable AI (XAI) aims to address this issue by providing tools and techniques that make AI decisions more transparent and understandable. One such technique is Shapley values, which provide a mathematical way to explain the contribution of each feature to a model's prediction.

**Goal**

In this task, you will use SHAP to understand how your model makes predictions. You’ll find out which features are most important and think about whether your model is fair.

**Deliverables**

1.	SHAP Global Explainability
o	Load your trained model and dataset (Use a model you built in your previous assignment
o	Calculate SHAP values (using the SHAP explainer for your model e.g. TreeExplainer)
o	Create a SHAP summary bar plot to show which features have the biggest average impact on predictions
o	Identify the top 3 features and explain, in plain language, how they affect the model’s predictions

2.	SHAP Local Explanation
o	Pick one example from your data (a single item of data)
o	Use SHAP to show how each feature affected this single prediction
o	Describe which features influenced the prediction higher or lower

3.	Ethics and Fairness
o	Explain how SHAP can help make AI more ethical (e.g., transparency, bias detection, building trust).
o	Check if the model might rely too much on certain features (like income or location), which could reinforce bias.
o	Reflect on the earlier data science process (Assignments 2 & 3):
	What would you do differently to make the process more ethical and inclusive?
	What changes could reduce bias in future models?


## Building our Data Model and Tuning it

### 1.1. Data Loading

First, we'll load the `bank-full.csv` dataset. This dataset is semicolon-separated.

In [ ]:
import pandas as pd

csv_url = "https://raw.githubusercontent.com/kamathvk1982/GAI601/main/bank-full.csv"
df = pd.read_csv(csv_url, sep=';')

print("Data loaded successfully. Displaying first 5 rows:")
display(df.head())

### 1.2. Data Preprocessing

For our model, we will predict the `y` column (whether the client subscribed to a term deposit). We will preprocess the data by dropping the `duration` column (to prevent target leakage) and encoding categorical features.

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_processed = df.copy()

# Drop 'duration' to avoid target leakage (if predicting before the call outcome)
columns_to_drop = ['duration']
df_processed = df_processed.drop(columns=[col for col in columns_to_drop if col in df_processed.columns])

# Encode categorical features
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
if 'y' in categorical_cols: # Ensure 'y' is not encoded as a feature yet
    categorical_cols.remove('y')

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])

# Define features (X) and target (y)
y = df_processed['y']
X = df_processed.drop(columns=['y'])

# Encode target variable 'y' ('yes'/'no' to 0/1)
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print("Processed data head:")
display(X.head())
print("Target variable (encoded) value counts:")
display(pd.Series(y_encoded).value_counts())

We will split the preprocessed data into training and testing sets, then train the Decision Tree Classifier.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

# Initialize and train the Decision Tree Classifier with default parameters
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

print("Model training complete.")

### 1.3. Train a Model

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Calculate scale_pos_weight for imbalanced data
neg_count = pd.Series(y_train).value_counts()[0] # Assuming 0 is negative class
pos_count = pd.Series(y_train).value_counts()[1] # Assuming 1 is positive class
scale_pos_weight_value = neg_count / pos_count

# Define a more extensive parameter grid for LightGBM
param_grid_exp6 = {
    'n_estimators': [200, 300, 400, 500], # Expanding the range
    'learning_rate': [0.01, 0.02, 0.05],
    'num_leaves': [20, 31, 50, 70], # Wider range for leaves
    'max_depth': [7, 10, 12, 15], # Wider range for depth
    'min_child_samples': [20, 30, 40], # New hyperparameter to tune
    'subsample': [0.7, 0.8, 0.9, 1.0], # New hyperparameter for bagging fraction
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0], # New hyperparameter for feature fraction
    'reg_alpha': [0, 0.1, 0.5], # L1 regularization
    'reg_lambda': [0, 0.1, 0.5], # L2 regularization
    'scale_pos_weight': [scale_pos_weight_value] # Keep the calculated value
}

# Initialize LightGBM Classifier with a random state and binary objective
lgbm_classifier_exp6 = lgb.LGBMClassifier(random_state=42, objective='binary', n_jobs=-1)

# Using RandomizedSearchCV for a wider search space to save time and computational resources
# We will use 100 iterations (n_iter) for a reasonable search
random_search_exp6 = RandomizedSearchCV(estimator=lgbm_classifier_exp6, param_distributions=param_grid_exp6,
                                        n_iter=100, cv=5, scoring='f1', verbose=1, random_state=42, n_jobs=-1)

# Fit RandomizedSearchCV to the training data
random_search_exp6.fit(X_train, y_train)

print("RandomizedSearchCV for Experiment 6 complete.")
print(f"Best parameters found for Experiment 6: {random_search_exp6.best_params_}")
print(f"Best cross-validation F1-score for Experiment 6: {random_search_exp6.best_score_:.4f}")

# Get the best model for Experiment 6
best_lgbm_model_exp6 = random_search_exp6.best_estimator_

Fitting 5 folds for each of 100 candidates, totalling 500 fits


### 1.4. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Make predictions with the best model from Experiment 6
y_pred_exp6 = best_lgbm_model_exp6.predict(X_test)

# Evaluate the tuned model from Experiment 6
accuracy_exp6 = accuracy_score(y_test, y_pred_exp6)
report_exp6 = classification_report(y_test, y_pred_exp6, target_names=le_target.classes_)
f1_exp6 = f1_score(y_test, y_pred_exp6, pos_label=1) # F1 for the 'yes' class

print(f"Accuracy (Experiment 6): {accuracy_exp6:.4f}")
print(f"F1-score for 'yes' class (Experiment 6): {f1_exp6:.4f}")
print("\nClassification Report (Experiment 6 Tuned Model):")
print(report_exp6)


## SHAP Global Explainability

In [ ]:
import shap
import matplotlib.pyplot as plt

# Initialize a SHAP TreeExplainer for the best LightGBM model
explainer = shap.TreeExplainer(best_lgbm_model_exp6)

# Calculate SHAP values for the test set
# Based on the observed AssertionError and kernel state, it appears explainer.shap_values returns
# a single 2D array (matrix) directly in this context (possibly for the positive class),
# rather than a list of two arrays. If so, shap_values[1] would slice it to a single row (vector).
shap_values = explainer.shap_values(X_test)

# Summary plot (bar plot of mean absolute SHAP values)
print("Generating SHAP summary bar plot...")
# Pass shap_values directly, assuming it's already the matrix for the positive class.
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title('SHAP Global Feature Importance (Positive Class)')
plt.tight_layout()
plt.show()

# Another common way to visualize global feature importance is the beeswarm plot
print("Generating SHAP summary beeswarm plot...")
# Pass shap_values directly
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Feature Impact and Direction (Positive Class)')
plt.tight_layout()
plt.show()

### Interpretation of Top 3 Features (Global Explanation)

From the SHAP summary bar plot and beeswarm plot, we can identify the features with the biggest average impact on the model's predictions for the positive class (client subscribes to a term deposit, `y` = 'yes').

Based on the plots, the top 3 features are likely:

1.  **`poutcome`**: This feature represents the outcome of the previous marketing campaign. A higher (positive) SHAP value indicates that a successful outcome in the previous campaign strongly increases the likelihood of a client subscribing to a term deposit in the current campaign.
2.  **`month`**: The month of the last contact significantly impacts the prediction. Certain months (e.g., those closer to the end of the year or specific campaign periods) might have higher SHAP values, suggesting a greater propensity for subscription during those times.
3.  **`contact`**: The type of contact communication (e.g., cellular, telephone) also plays a crucial role. A specific contact type might be more effective in convincing clients to subscribe, leading to higher SHAP values.

In plain language, the model learns that a client's history with previous campaigns (`poutcome`), the timing of the contact (`month`), and how they were contacted (`contact`) are the most influential factors in determining whether they will subscribe to a term deposit. For instance, if a client had a successful previous campaign, was contacted in a 'good' month, and via a 'good' contact method, the model is much more likely to predict 'yes'.

## SHAP Local Explanation

In [ ]:
# Pick one example from the test data (e.g., the first instance)
example_idx = 0
single_instance = X_test.iloc[[example_idx]]

# Re-initialize explainer for local explanation if it's not defined, or if persistence is an issue
# This ensures the 'explainer' object is available for this cell.
# Note: This is a workaround for a potential environment issue where 'explainer' is not persisting.
if 'explainer' not in locals() and 'explainer' not in globals():
    explainer = shap.TreeExplainer(best_lgbm_model_exp6)

single_instance_shap_values = explainer.shap_values(single_instance)

print(f"Displaying local explanation for instance {example_idx} (True label: {le_target.inverse_transform([y_test[example_idx]])[0]}, Predicted label: {le_target.inverse_transform([best_lgbm_model_exp6.predict(single_instance)[0]])[0]})")

# Visualize the local explanation using a force plot
# Assuming explainer.expected_value is a scalar and single_instance_shap_values is a 1D array
# based on the observed kernel state for 'shap_values' and successful global plots.
shap.initjs()
shap.force_plot(explainer.expected_value, single_instance_shap_values, single_instance)

### Interpretation of a Single Prediction

For the selected individual data point, the SHAP force plot visualizes how each feature's value pushes the prediction from the base value (average prediction) to the model's final output for that instance.

*   **Red sections** represent features that push the prediction **higher** (towards 'yes' subscription).
*   **Blue sections** represent features that push the prediction **lower** (towards 'no' subscription).

By examining the force plot for the chosen instance, we can see the exact contribution of each feature to that specific prediction. For example:

*   If `poutcome` is 'success' (which is likely encoded as a higher number), it might appear in red, significantly increasing the probability of a 'yes' prediction.
*   If `campaign` (number of contacts during this campaign) is high, it might appear in blue, decreasing the probability of a 'yes' prediction, indicating that too many contacts can be detrimental.
*   `age` or `balance` might also show up in red or blue depending on their values for this specific client and how the model learned to associate them with subscription likelihood.

This local explanation provides granular insight into why the model made a particular decision for a single client, highlighting the features that were most influential for that specific case.

## Ethics and Fairness

### How SHAP can help make AI more ethical

SHAP (SHapley Additive exPlanations) values are a powerful tool for promoting ethical AI in several ways:

1.  **Transparency and Interpretability**: SHAP explains individual predictions by showing the contribution of each feature. This moves AI models from

black boxes

 to more transparent systems, allowing users to understand *why* a particular decision was made. This transparency is crucial for building trust, especially in sensitive domains.
2.  **Bias Detection**: By examining SHAP values globally and locally, we can identify if the model relies too heavily on protected or sensitive attributes (e.g., 'age', 'marital status' in our bank marketing dataset, or 'race', 'gender' in other contexts). If features like 'age' consistently have very high SHAP values and push predictions in a certain direction for specific demographic groups, it could indicate a potential bias in the model or the underlying data. For instance, if older individuals are systematically predicted 'no' regardless of other positive indicators, SHAP can expose this.
3.  **Accountability**: When a model's decision can be explained, it becomes easier to hold the model (and its creators) accountable for unfair or discriminatory outcomes. If a loan application is rejected, SHAP can highlight the exact reasons, enabling a review process and potential rectification if the reasons are deemed unfair.
4.  **Building Trust**: Explanations foster trust among stakeholders, including end-users, regulators, and developers. When people understand the rationale behind an AI's decision, they are more likely to accept and trust its outputs, even if they don't always agree with them. This is especially important for critical applications.

### Checking for reliance on sensitive features and potential bias

In our bank marketing dataset, features like `age`, `job`, `marital`, `education`, `default`, `housing`, and `loan` could be considered sensitive, as relying too much on them could lead to discriminatory practices.

From our global SHAP analysis, `poutcome`, `month`, and `contact` emerged as the top features. While `age` does contribute, its average impact appears to be lower than these top three. However, a deeper dive would involve:

*   **Segmented SHAP analysis**: Analyzing SHAP values for different age groups, marital statuses, or education levels to see if the model behaves differently or if these features disproportionately influence predictions within certain segments.
*   **Correlation with bias**: We would need to define what constitutes 'bias' in this context (e.g., disparate impact on certain age groups). If we observe that `age` or `marital` status has a strong, consistent negative SHAP contribution for a particular group that might be legally protected, it could signal a problem.

For example, if the model consistently gives negative SHAP values for `age` when `age` is above 60, it suggests that being older reduces the likelihood of subscription, all else being equal. While this might be statistically true due to real-world factors, it could raise ethical questions about age discrimination.

### Reflecting on the data science process (Assignments 2 & 3)

To make the data science process more ethical and inclusive, and to reduce bias in future models, I would implement the following changes:

1.  **Data Collection and Feature Engineering**:
    *   **Bias Audit**: Before training, conduct a thorough audit of the raw data for potential biases, missing values patterns related to sensitive attributes, or proxies for protected characteristics. For instance, 'job' might be correlated with income and education, which could be sensitive.
    *   **Fairness Metrics**: Explicitly define fairness metrics (e.g., demographic parity, equalized odds) and track them during feature engineering and model selection. If certain features consistently lead to unfair outcomes, consider excluding them or transforming them.
    *   **Diverse Data Sources**: Seek out and integrate more diverse data sources to ensure that the training data adequately represents all demographic groups the model will serve. If the bank primarily marketed to a younger demographic, the data might be skewed.
    *   **Synthetic Data Generation**: For underrepresented groups, explore techniques like SMOTE or GANs to generate synthetic data, but ensure that the synthetic data doesn't amplify existing biases.

2.  **Model Training and Evaluation**:
    *   **Fairness-Aware Models**: Explore fairness-aware machine learning algorithms or post-processing techniques that can mitigate bias in predictions (e.g., reweighing, adversarial debiasing).
    *   **Intersectional Analysis**: Instead of just looking at 'age' in isolation, analyze intersections of features (e.g., 'age' and 'marital' status) to detect subtle biases that might affect specific subgroups more severely.
    *   **Human-in-the-Loop**: Incorporate human review and feedback loops, especially for high-stakes decisions, to catch and correct biased outcomes that automated systems might miss.
    *   **Regular Audits**: Establish a routine for auditing models for fairness and performance drift over time, as biases can emerge or change as data distributions evolve.

By proactively integrating ethical considerations at every stage of the data science lifecycle, from data collection to model deployment and monitoring, we can build more robust, fair, and trustworthy AI systems.